# Using a Custom Model



---
## 1. Setup & Configuration
Import necessary modules and set the working directory to the repository root.

In [ ]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython

from IPython.display import display
import os 

# replace this with the path to the directory on your machine
# os.chdir('../..')
os.chdir('/media/haupert/data/mes_projets/_packages/bacpipe.git/bacpipe')

# import the package
import bacpipe

---
## 2. Create a new model from scratch
Define your own model by subclassing `ModelBaseClass` and plug it directly into bacpipe's pipeline.

In [ ]:
import librosa as lb
from bacpipe.model_pipelines.model_utils import ModelBaseClass

class MyModel(ModelBaseClass):
    SAMPLE_RATE = 48000         # the sample rate of the audio files that will be processed by the model
    SEGMENT_LENGTH = 48000*3    # 3s => the length of the audio segments that will be processed by the model (in samples)

    def __init__(self, **kwargs):
        super().__init__(sr=self.SAMPLE_RATE, segment_length=self.SEGMENT_LENGTH, **kwargs)

    def preprocess(self, audio):
        return audio

    def __call__(self, audio):
        audio = audio.cpu().numpy()
        mel_spec = lb.feature.melspectrogram(y=audio, sr=self.SAMPLE_RATE)
        # return array needs to be 2D!
        mel_spec = mel_spec.reshape(
            [len(mel_spec), mel_spec.shape[-2] * mel_spec.shape[-1]]
            )
        return mel_spec

---
## 3. Using this new built-in model

1. First generate the feature vectors (which are mels in this case, and not the embeddings of a deeplearning model)
2. Train a probe to evaluate the quality of the feature vectors (or embeddings) of the model

In [ ]:
# load the data to be process as well the model and compute the embeddings
loader_obj = bacpipe.run_pipeline_for_single_model(
    model_name='mel',                               # name of the model to run. Supported models are in bacpipe.supported_models
    audio_dir='bacpipe/tests/test_data',                # path to directory containing audio files  
    CustomModel=MyModel
)

# get the computed embeddings as an array
embeds = loader_obj.embeddings(return_type='array')

In [ ]:
# Run this function after computing the embeddings otherwise it is no able to find the connection between embeddings and labels
gt = bacpipe.ground_truth_by_model(
    'mel', 
    'bacpipe/tests/test_data', 
    annotations_filename='annotations.csv',
    single_label=False,
    overwrite=False
)

# Train and test a new probe associated to the feature vectors of the model 
# and evaluate the performance of the probe. 
# The returned metrics are the same as for the evaluation of a classification model, 
# but here they are used to evaluate the quality of the embeddings of the model.
probe, label2idx, metrics = bacpipe.probing_pipeline(
    model_name='mel', 
    ground_truth=gt,
    embeds=embeds)

# display the main metrics of the probe
display(metrics['overall'])